In [53]:
import warnings
warnings.filterwarnings('ignore')

import os
import sys
import glob
import time
import datetime
import requests
import argparse
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import AutoMinorLocator
import matplotlib as mpl
from spacepy import pycdf
from scipy.ndimage import gaussian_filter
from astropy.visualization import ImageNormalize, PercentileInterval
from matplotlib.ticker import ScalarFormatter

mpl.rcParams['date.epoch'] = '1970-01-01T00:00:00'
try:
    mdates.set_epoch('1970-01-01T00:00:00')
except:
    pass

plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['savefig.facecolor'] = 'white'

rootdir = '/home/mnedal/data'

In [24]:
def fetch_STAswaves(year=None, month=None, day=None):
    '''
    Download STEREO/SWAVES data.
        Inputs: year, month, day.
    '''
    try:
        os.makedirs(f'{rootdir}/stereo_data/', exist_ok=True)
    except:
        pass
    
    file_path = f'{rootdir}/stereo_data/stereo_level2_swaves_{year}{month}{day}_v02.cdf'

    if not os.path.exists(file_path):
        # get the data file
        URL = f'https://spdf.gsfc.nasa.gov/pub/data/stereo/combined/swaves/level2_cdf/{year}/stereo_level2_swaves_{year}{month}{day}_v02.cdf'
        response = requests.get(URL)
        open(os.path.join(f'{rootdir}/stereo_data', f'stereo_level2_swaves_{year}{month}{day}_v02.cdf'), 'wb').write(response.content)
    
    # read the data values
    cdf_stereo = pycdf.CDF(f'{rootdir}/stereo_data/stereo_level2_swaves_{year}{month}{day}_v02.cdf')
    time_ste = np.array(cdf_stereo.get('Epoch'))
    freq_ste = np.array(cdf_stereo.get('frequency'))
    data_ste_A = np.array(cdf_stereo.get('avg_intens_ahead'))

    # calculate the mean value in each row (freq channel)
    # and subtract it from each corresponding row
    df_ste_A = pd.DataFrame(data_ste_A.T)
    df_ste_mean = df_ste_A.mean(axis=1)
    data_ste_A = df_ste_A.sub(df_ste_mean, axis=0)

    # Apply Gaussian smoothing
    smoothed_ste_A = gaussian_filter(data_ste_A, sigma=1)

    ste_norm = ImageNormalize(smoothed_ste_A, interval=PercentileInterval(98))#, clip=True)

    return time_ste, freq_ste, smoothed_ste_A, ste_norm

In [25]:
date = '2024-05-14'

YEAR, MONTH, DAY = date.split('-')
time_ste, freq_ste, data_ste_A, ste_norm = fetch_STAswaves(year=YEAR, month=MONTH, day=DAY)

In [ ]:
fig = plt.figure(figsize=[15,5])
ax = fig.add_subplot(111)
ax.pcolormesh(time_ste,
               freq_ste/1e3,
               data_ste_A,
               norm=ste_norm,
               cmap='RdYlBu_r')
ax.set_yscale('log')
ax.set_ylim(ax.get_ylim()[::-1]) # Reverse the y-axis
ax.grid(False)
ax.set_xlabel('Time (UT)')
ax.set_ylabel('Frequency (MHz)')
ax.set_title(f"STEREO/SWAVES: {freq_ste[0]:.2f} kHz $-$ {freq_ste[-1]/1e3:.2f} MHz")
ax.xaxis_date()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
# ax.yaxis.set_major_formatter(ScalarFormatter())
fig.savefig(f'{rootdir}/stereo_data/stereo_swaves_{date}.png', format='png', bbox_inches='tight')
plt.show()

In [ ]:
start_time = '16:35'
end_time = '17:50'

fig = plt.figure(figsize=[12,6])
ax = fig.add_subplot(111)
ax.pcolormesh(time_ste,
               freq_ste/1e3,
               data_ste_A,
               norm=ste_norm,
               cmap='RdYlBu_r')
ax.set_yscale('log')
# Define the custom ticks
custom_ticks = [freq_ste[0]*1e-3, 4e-3, 6e-3, 8e-3, 10e-3,
                2e-2, 4e-2, 6e-2, 8e-2, 10e-2,
                2e-1, 4e-1, 6e-1, 8e-1, 10e-1,
                2, 4, 6, 8, 10, 12, 14, freq_ste[-1]/1e3]
ax.set_yticks(custom_ticks)
ax.set_yticklabels([str(f'{tick:.3f}') for tick in custom_ticks])
ax.xaxis.set_minor_locator(AutoMinorLocator(n=10))
ax.set_ylim(ax.get_ylim()[::-1]) # Reverse the y-axis
ax.grid(False)
ax.set_xlabel('Time (UT)')
ax.set_ylabel('Frequency (MHz)')
# ax.set_title(f'STEREO/SWAVES: {freq_ste[0]:.2f} kHz $-$ {freq_ste[-1]/1e3:.2f} MHz between {start_time} $-$ {end_time} UT')
ax.xaxis_date()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax.set_xlim(left=pd.Timestamp(f'{date} {start_time}'), right=pd.Timestamp(f'{date} {end_time}'))
# ax.yaxis.set_major_formatter(ScalarFormatter())
# fig.savefig(f'{rootdir}/stereo_data/stereo_swaves_{date}_{start_time}_{end_time}.png', format='png', bbox_inches='tight')
plt.show()
print(f'STEREO/SWAVES: {freq_ste[0]:.2f} kHz - {freq_ste[-1]/1e3:.2f} MHz between {start_time} - {end_time} UT')

In [84]:
time_ste[1], time_ste[0]

(datetime.datetime(2024, 5, 14, 0, 1, 30),
 datetime.datetime(2024, 5, 14, 0, 0, 30))